In [ ]:
# Pipeline to process and integrate ICD classification, lab data, and chart extraction results for analysis.

# Cell 1: Import necessary libraries and load raw data files
import pandas as pd

# Define file paths (adjust to your environment)
EXTRACTION_FILE  = 'ICD_Team_Output.xlsx'    #only change required...from icd team
CHART_FILE       = 'IP_data_extraction_final_01June2026.xlsx'
LABS_FILE        = 'Lab_extracted_per_chart_complete_oss_18june.xlsx'
ICD_CLASS_FILE   = 'Acute_or_chronic_database.xlsx'
LAB_DATABASE_FILE = 'Final_Lab_database.csv'

# Load raw data (no parsing, no helpers)
print("Loading raw data files...")
raw_icd_class_df1 = pd.read_excel(ICD_CLASS_FILE, sheet_name="Sheet1", engine='openpyxl')
raw_icd_class_df2 = pd.read_excel(ICD_CLASS_FILE, sheet_name="Sheet2", engine='openpyxl')
raw_lab_database_df = pd.read_csv(LAB_DATABASE_FILE, encoding='latin-1', low_memory=False)
raw_extracted_labs_df = pd.read_excel(LABS_FILE)
raw_chart_data_df = pd.read_excel(CHART_FILE)
raw_extraction_df = pd.read_excel(EXTRACTION_FILE) if EXTRACTION_FILE.endswith('.xlsx') else pd.read_csv(EXTRACTION_FILE, encoding='latin-1')

print("All raw data loaded.")

Loading raw data files...
All raw data loaded.


In [ ]:
# Cell 2: Define helper functions for normalization and mapping

import json
import ast
import re
from collections import defaultdict

# ------------------- Helper functions -------------------
def ensure_decimal(code):
    if not isinstance(code, str):
        code = str(code)
    if '.' in code:
        return code
    if len(code) <= 3:
        return code
    return code[:3] + '.' + code[3:]

def normalize_lab_name(name):
    if not isinstance(name, str):
        return ""
    name = name.lower().strip()
    name = re.sub(r'[^a-z0-9\s\(\)/\.\+\-]', ' ', name)
    name = re.sub(r'\s+', ' ', name).strip()
    name = re.sub(r'\b(lvl|level|value|result|relative|absolute)\b', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    
    # --- UPDATE THIS DICTIONARY AS NEEDED ---
    LAB_SYNONYMS = {
        # ── CBC core ─────────────────────────────────────────────
        "wbc": "white blood cells",
        "white blood cells": "white blood cells",
        "white blood cell (wbc) count": "white blood cells",
        "white blood cell count": "white blood cells",
        "white blood cell count (wbc)": "white blood cells",
        "white blood cells (wbc)": "white blood cells",
        "complete blood count (cbc)": "complete blood count",
        "complete blood count (cbc) with differential": "complete blood count",
        "complete blood count (cbc) with differential leukocyte count": "complete blood count",
        "complete blood count": "complete blood count",
        "cbc": "complete blood count",

        "rbc": "red blood cells",
        "red blood cells": "red blood cells",
        "red blood cell (rbc) count": "red blood cells",

        "hemoglobin": "hemoglobin",
        "hgb": "hemoglobin",
        "hb": "hemoglobin",
        "hemoglobin (hgb)": "hemoglobin",
        "hemoglobin (hb)": "hemoglobin",
        "hemoglobin (hgb)": "hemoglobin",
        "fetal hemoglobin (hb)": "hemoglobin",
        "fetal hemoglobin (hbf)": "hemoglobin",

        "hematocrit": "hematocrit",
        "hct": "hematocrit",
        "hematocrit (hct)": "hematocrit",
        "hematocrit (hct)": "hematocrit",

        "platelets": "platelets",
        "plt": "platelets",
        "platelet count": "platelets",
        "platelet count": "platelets",
        "platelets (estimate)": "platelets",
        "plt estimate": "platelets",
        "pit estimate": "platelets",

        "mean corpuscular volume": "mean corpuscular volume",
        "mcv": "mean corpuscular volume",

        "mean corpuscular hemoglobin": "mean corpuscular hemoglobin",
        "mch": "mean corpuscular hemoglobin",

        "mean corpuscular hemoglobin concentration": "mean corpuscular hemoglobin concentration",
        "mchc": "mean corpuscular hemoglobin concentration",

        "rdw": "rdw",
        "mpv": "mpv",

        # ── CBC differentials ────────────────────────────────────
        "neutrophils": "neutrophils",
        "neutrophils (absolute)": "neutrophils",
        "neutrophils (manual)": "neutrophils",
        "neutrophils (relative)": "neutrophils",
        "neutrophils absolute": "neutrophils",
        "neutrophil man": "neutrophils",
        "neutrophil segmented": "neutrophils",
        "neutrophil count": "neutrophils",
        "neutrophil count (absolute)": "neutrophils",
        "absolute neutrophil count (anc)": "neutrophils",
        "segs man": "neutrophils",

        "lymphocytes": "lymphocytes",
        "lymphocytes (absolute)": "lymphocytes",
        "lymphocytes (manual)": "lymphocytes",
        "lymphocytes (relative)": "lymphocytes",
        "lymphocytes absolute": "lymphocytes",
        "lymphocyte man": "lymphocytes",
        "absolute lymphocyte count (alc)": "lymphocytes",

        "monocytes": "monocytes",
        "monocytes (absolute)": "monocytes",
        "monocytes (manual)": "monocytes",
        "monocytes (relative)": "monocytes",
        "monocytes absolute": "monocytes",
        "monocytes (relat": "monocytes",
        "monocyte man": "monocytes",

        "eosinophils": "eosinophils",
        "eosinophils (absolute)": "eosinophils",
        "eosinophils (manual)": "eosinophils",
        "eosinophils (relative)": "eosinophils",
        "eosinophils absolute": "eosinophils",
        "eos man": "eosinophils",
        "eosinophil count": "eosinophils",
        "eosinophil count (absolute)": "eosinophils",
        "eosinophilia": "eosinophils",
        "eosinophils (": "eosinophils",

        "basophils": "basophils",
        "basophils (absolute)": "basophils",
        "basophils (manual)": "basophils",
        "basophils (relative)": "basophils",
        "basophils absolute": "basophils",
        "basophil man": "basophils",
        "basophil count": "basophils",

        "band neutrophils": "band neutrophils",
        "band neutrophils (relative)": "band neutrophils",
        "band man": "band neutrophils",

        "metamyelocytes": "metamyelocytes",
        "metamyelo man": "metamyelocytes",
        "met": "metamyelocytes",

        "myelocytes": "myelocytes",
        "myelocyte man": "myelocytes",

        "atypical lymphocytes": "atypical lymphocytes",
        "atypical lymphocytes (absolute)": "atypical lymphocytes",
        "atypical lymphocytes (manual)": "atypical lymphocytes",

        "nrbc": "nrbc",
        "nrbc (absolute)": "nrbc",
        "nrbc (relative)": "nrbc",
        "nrbc man": "nrbc",

        "cells counted": "cells counted",
        "total cells counted manually": "cells counted",

        # ── BMP / CMP electrolytes ───────────────────────────────
        "sodium": "sodium",
        "sodium lvl": "sodium",
        "serum sodium": "sodium",
        "plasma sodium": "sodium",
        "na": "sodium",
        "electrolytes": "electrolytes",
        "serum electrolytes": "electrolytes",
        "electrolyte panel": "electrolytes",
        "electrolytes panel": "electrolytes",
        "basic metabolic panel": "basic metabolic panel",
        "bmp": "basic metabolic panel",
        "comprehensive metabolic panel": "comprehensive metabolic panel",
        "cmp": "comprehensive metabolic panel",
        "complete metabolic panel (cmp)": "comprehensive metabolic panel",

        "potassium": "potassium",
        "potassium lvl": "potassium",
        "serum potassium": "potassium",
        "potassium, serum": "potassium",
        "k": "potassium",
        "serum k": "potassium",

        "chloride": "chloride",
        "chloride lvl": "chloride",
        "serum chloride": "chloride",
        "cl": "chloride",

        "carbon dioxide": "carbon dioxide",
        "co2": "carbon dioxide",
        "c02": "carbon dioxide",
        "pco2": "carbon dioxide",
        "pco2": "carbon dioxide",
        "pc02": "carbon dioxide",
        "paco2": "carbon dioxide",
        "bicarbonate": "carbon dioxide",

        "bun": "bun",
        "blood urea nitrogen": "bun",
        "blood urea nitrogen (bun)": "bun",
        "serum bun": "bun",
        "bun/creatinine ratio": "bun/creatinine ratio",
        "bun/creatinine": "bun/creatinine ratio",
        "bun/creat": "bun/creatinine ratio",
        "bun/creatine ratio": "bun/creatinine ratio",

        "creatinine": "creatinine",
        "creatinine lvl": "creatinine",
        "serum creatinine": "creatinine",
        "creatinine (cr)": "creatinine",
        "creatinine (serum)": "creatinine",
        "creatinine, serum": "creatinine",
        "cr": "creatinine",

        "glucose": "glucose",
        "glucose level": "glucose",
        "blood glucose": "glucose",
        "serum glucose": "glucose",
        "glucose poc accu-chek": "glucose",
        "glucose poc accu-chek.": "glucose",
        "random plasma glucose": "glucose",
        "plasma glucose": "glucose",
        "blood glucose level": "glucose",

        "calcium": "calcium",
        "calcium lvl": "calcium",
        "serum calcium": "calcium",
        "calcium level": "calcium",
        "calcium, serum": "calcium",
        "ca": "calcium",

        "albumin": "albumin",
        "albumin lvl": "albumin",
        "serum albumin": "albumin",

        "total protein": "total protein",
        "protein, total": "total protein",
        "serum protein and albumin": "total protein",

        
        "hco3": "bicarbonate",
        "serum bicarbonate": "carbon dioxide",

        "phosphate": "phosphate",
        "phosphorus": "phosphate",
        "serum phosphorus": "phosphate",
        "serum phosphate": "phosphate",

        "magnesium": "magnesium",
        "magnesium level": "magnesium",
        "magnesium lvl": "magnesium",
        "serum magnesium": "magnesium",
        "mg": "magnesium",
        "serum magnesium": "magnesium",

        "anion gap": "anion gap",
        "serum anion gap": "anion gap",
        "agap": "anion gap",

        "osmolality": "osmolality",
        "osmolality calc": "osmolality",
        "serum osmolality": "osmolality",
        "osmolarity": "osmolality",
        "serum osmol gap": "serum osmolal gap",
        "serum osmolal gap": "serum osmolal gap",
        "osmolal gap": "serum osmolal gap",

        "a/g ratio": "albumin/globulin ratio",
        "globulin": "globulin",
        "ag": "albumin/globulin ratio",

        # ── LFTs ─────────────────────────────────────────────────
        "ast": "ast",
        "alt": "alt",
        "ast/alt ratio": "ast/alt ratio",
        "alanine aminotransferase (alt)": "alt",
        "aspartate aminotransferase (ast)": "ast",

        "alkaline phosphatase": "alkaline phosphatase",
        "alk phos": "alkaline phosphatase",
        "alkaline phosphatase (alp)": "alkaline phosphatase",
        "alp": "alkaline phosphatase",

        "total bilirubin": "total bilirubin",
        "bili total": "total bilirubin",
        "bill total": "total bilirubin",
        "bilirubin, total": "total bilirubin",
        "bilirubin (total)": "total bilirubin",
        "bilirubin (total and direct)": "total bilirubin",
        "bilirubin (total and indirect)": "total bilirubin",
        "total serum bilirubin": "total bilirubin",
        "total serum bilirubin (tsb)": "total bilirubin",
        "bilirubin, total": "total bilirubin",
        "serum bilirubin (total and direct)": "total bilirubin",

        "direct bilirubin": "direct bilirubin",
        "bili direct": "direct bilirubin",
        "direct serum bilirubin": "direct bilirubin",
        "direct serum bilirubin (conjugated)": "direct bilirubin",

        "indirect bilirubin": "indirect bilirubin",

        "liver function tests": "liver function tests",
        "lfts": "liver function tests",
        "lft": "liver function tests",
        "liver function panel (lfts)": "liver function tests",
        "liver panel": "liver function tests",
        "liver function tests (lfts)": "liver function tests",

        "hepatic steatosis": "hepatic steatosis",

        # ── Lipids ───────────────────────────────────────────────
        "total cholesterol": "total cholesterol",
        "chol": "total cholesterol",
        "ldl cholesterol": "ldl cholesterol",
        "ldl": "ldl cholesterol",
        "ldl-c": "ldl cholesterol",
        "low-density lipoprotein cholesterol": "ldl cholesterol",
        "low-density lipoprotein cholesterol (ldl-c)": "ldl cholesterol",
        "hdl cholesterol": "hdl cholesterol",
        "hdl": "hdl cholesterol",
        "triglycerides": "triglycerides",
        "tg": "triglycerides",
        "lipid panel": "lipid panel",
        "lipoprotein(a) [lp(a)]": "lipoprotein(a)",
        "chylomicrons": "chylomicrons",

        # ── Coagulation ──────────────────────────────────────────
        "pt": "pt",
        "pt/inr": "pt/inr",
        "prothrombin time": "pt",
        "prothrombin time (pt)": "pt",
        "prothrombin time/international normalized ratio (pt/inr)": "pt/inr",

        "inr": "inr",
        "international normalized ratio (inr)": "inr",

        "ptt": "ptt",
        "aptt": "ptt",
        "partial thromboplastin time (ptt)": "ptt",
        "activated partial thromboplastin time (aptt)": "ptt",

        "fibrinogen": "fibrinogen",
        "fibrinogen level": "fibrinogen",

        "d-dimer": "d-dimer",
        "d dimer": "d-dimer",

        "coagulation panel": "coagulation panel",
        "coagulation studies": "coagulation panel",
        "coagulation profile": "coagulation panel",

        "bleeding time": "bleeding time",
        "lupus anticoagulant": "lupus anticoagulant",
        "anticardiolipin antibodies (igg, igm)": "anticardiolipin antibodies",
        "antiphospholipid antibodies (apl)": "antiphospholipid antibodies",

        # ── Cardiac markers ──────────────────────────────────────
        "troponin": "troponin",
        "troponin i": "troponin",
        "troponin i hs": "troponin",
        "troponin i h.s.": "troponin",
        "troponin i or t": "troponin",
        "troponin / ck-mb": "troponin",
        "troponin, ck-mb": "troponin",
        "cardiac troponin": "troponin",
        "cardiac troponin i or t": "troponin",
        "cardiac enzymes": "cardiac enzymes",
        "cardiac enzymes (troponin)": "cardiac enzymes",

        "bnp": "bnp",
        "bnp (b-type natriuretic peptide)": "bnp",
        "bnp (b-type natriuretic peptide)": "bnp",
        "bnp or nt-probnp": "bnp",
        "bnp / nt-probnp": "bnp",
        "bnp/nt-probnp": "bnp",
        "bnp-probnp": "bnp",
        "brain natriuretic peptide": "bnp",
        "brain natriuretic peptide (bnp)": "bnp",
        "b-type natriuretic peptide (bnp)": "bnp",
        "b-type natriuretic peptide (bnp) or n-terminal pro-bnp (nt-probnp)": "bnp",

        "nt-probnp": "nt-probnp",
        "n-terminal pro-b-type natriuretic peptide (nt-probnp)": "nt-probnp",

        "creatine kinase": "creatine kinase",
        "ck": "creatine kinase",
        "ck total": "creatine kinase",
        "creatinine kinase": "creatine kinase",
        "creatinine kinase (ck)": "creatine kinase",
        "creatine kinase (ck)": "creatine kinase",

        "ck-mb": "ck-mb",

        "ejection fraction": "ejection fraction",
        "ef/ ejection fraction": "ejection fraction",
        "ef/ejection fraction": "ejection fraction",

        "qt interval": "qt interval",
        "qtc interval": "qt interval",
        "qtc": "qt interval",
        "qrs duration": "qrs duration",
        "qrs duration on ecg": "qrs duration",

        # ── Thyroid ──────────────────────────────────────────────
        "tsh": "tsh",
        "tsh test": "tsh",
        "thyroid stimulating hormone": "tsh",
        "thyroid-stimulating hormone (tsh)": "tsh",
        "thyroid stimulating hormone (tsh)": "tsh",
        "serum tsh": "tsh",

        "free t4": "free t4",
        "t4 free": "free t4",
        "serum free t4": "free t4",
        "free thyroxine (free t4)": "free t4",
        "free thyroxine (t4)": "free t4",

        "free t3": "free t3",

        "total t4": "total t4",
        "t4": "total t4",
        "total thyroxine (total t4)": "total t4",

        "total t3": "total t3",
        "t3 total": "total t3",

        "thyroglobulin": "thyroglobulin",
        "thyroid function tests": "thyroid function tests",
        "thyroid function tests (tfts)": "thyroid function tests",
        "tfts": "thyroid function tests",

        # ── Hormones ─────────────────────────────────────────────
        "fsh": "fsh",
        "follicle-stimulating hormone (fsh)": "fsh",
        "serum fsh": "fsh",

        "lh": "lh",
        "luteinizing hormone (lh)": "lh",

        "prolactin": "prolactin",
        "prl": "prolactin",
        "prolactin level": "prolactin",
        "serum prolactin": "prolactin",

        "testosterone": "testosterone",
        "total testosterone": "testosterone",
        "testosterone, total": "testosterone",
        "testosterone (total)": "testosterone",
        "total t": "testosterone",
        "free testosterone": "testosterone",
        "testosterone levels": "testosterone",

        "estradiol": "estradiol",
        "estradiol (e2)": "estradiol",

        "progesterone": "progesterone",

        "acth": "acth",
        "acth (adrenocorticotropic hormone)": "acth",
        "acth, plasma": "acth",
        "plasma acth": "acth",
        "adrenocorticotropic hormone (acth)": "acth",

        "cortisol": "cortisol",
        "cortisol level": "cortisol",
        "serum cortisol": "cortisol",
        "cortisol, serum": "cortisol",
        "cortisol, serum, am": "cortisol",
        "cortisol, free, urine, 24 hour": "24-hour urine free cortisol",
        "morning serum cortisol": "cortisol",

        "growth hormone (gh)": "growth hormone",
        "igf-1": "igf-1",
        "insulin-like growth factor 1 (igf-1)": "igf-1",

        "amh": "amh",

        "aldosterone": "aldosterone",
        "aldosterone-to-renin ratio (arr)": "aldosterone-to-renin ratio",

        "parathyroid hormone": "parathyroid hormone",
        "pth": "parathyroid hormone",
        "serum parathyroid hormone": "parathyroid hormone",
        "parathyroid hormone (pth)": "parathyroid hormone",

        "calcitonin": "calcitonin",
        "renin": "renin",
        "glucagon": "glucagon",
        "insulin": "insulin",
        "c-peptide": "c-peptide",

        # ── Diabetes ─────────────────────────────────────────────
        "hemoglobin a1c": "hemoglobin a1c",
        "hemoglobin a1c (hba1c)": "hemoglobin a1c",
        "hba1c": "hemoglobin a1c",
        "hba1c": "hemoglobin a1c",
        "hgb ale": "hemoglobin a1c",
        "estimated average glucose (eag)": "estimated average glucose",
        "eag": "estimated average glucose",

        "fasting plasma glucose": "fasting plasma glucose",
        "fasting blood glucose": "fasting plasma glucose",
        "glucose tolerance test (gtt)": "glucose tolerance test",
        "oral glucose tolerance test": "glucose tolerance test",

        # ── Renal ────────────────────────────────────────────────
        "egfr": "egfr",
        "egfr ckd-epi": "egfr",
        "estimated glomerular filtration rate (egfr)": "egfr",
        "creatinine clearance": "creatinine clearance",
        "cystatin c": "cystatin c",
        "uric acid": "uric acid",
        "serum uric acid": "uric acid",
        "blood uric acid": "uric acid",
        "urine output": "urine output",
        "urine osmolality": "urine osmolality",
        "urine sodium": "urine sodium",
        "urine protein": "urine protein",
        "ua protein": "urine protein",
        "urine protein-to-creatinine ratio": "urine protein-to-creatinine ratio",
        "urine protein creatinine ratio": "urine protein-to-creatinine ratio",
        "urine protein creatinine ratio (upcr)": "urine protein-to-creatinine ratio",
        "urine protein/creatinine ratio": "urine protein-to-creatinine ratio",
        "urine protein to creatinine ratio": "urine protein-to-creatinine ratio",
        "urine protein, total": "urine protein",
        "microalbuminuria": "urine albumin-to-creatinine ratio",
        "urine albumin-to-creatinine ratio": "urine albumin-to-creatinine ratio",
        "microalbuminuria / urine albumin-to-creatinine ratio": "urine albumin-to-creatinine ratio",
        "kidney function tests": "renal function tests",
        "renal function tests": "renal function tests",
        "renal function panel (bun, creatinine)": "renal function tests",
        "24-hour urine protein": "24-hour urine protein",

        # ── Urinalysis ───────────────────────────────────────────
        "urinalysis": "urinalysis",
        "urinalysis (ua)": "urinalysis",
        "urinanalysis": "urinalysis",
        "urine analysis": "urinalysis",
        "ua color": "ua color",
        "ua appear": "ua appearance",
        "ua ph": "ua ph",
        "ua spec grav": "ua specific gravity",
        "ua glucose": "ua glucose",
        "ua blood": "ua blood",
        "ua bili": "ua bilirubin",
        "ua bili pba probetec": "ua bilirubin",
        "ua nitrite": "ua nitrite",
        "ua leuk est": "ua leukocyte esterase",
        "ua wbc": "ua wbc",
        "ua rbc": "ua rbc",
        "ua bacteria": "ua bacteria",
        "ua bacteria": "ua bacteria",
        "ua urobilinogen": "ua urobilinogen",
        "ua squam epith": "ua squamous epithelial cells",
        "ua mucus": "ua mucus",
        "ua ketones": "urine ketones",
        "ua ketones pba probetec": "urine ketones",
        "urine ketones": "urine ketones",
        "urine drug screen": "urine drug screen",
        "urine drug screen (uds)": "urine drug screen",
        "urine culture": "urine culture",
        "urine culture and sensitivity": "urine culture and sensitivity",
        "urinary culture": "urine culture",
        "post-void residual (pvr)": "post-void residual",
        "post void residual (pvr)": "post-void residual",

        # ── Iron / Vitamins ──────────────────────────────────────
        "ferritin": "ferritin",
        "serum ferritin": "ferritin",
        "iron studies": "iron studies",
        "serum iron": "iron studies",
        "serum iron studies": "iron studies",
        "transferrin saturation": "transferrin saturation",
        "tibc": "tibc",

        "vitamin d": "25-hydroxyvitamin d",
        "25-hydroxyvitamin d": "25-hydroxyvitamin d",
        "25-hydroxy vitamin d": "25-hydroxyvitamin d",
        "vitamin d level": "25-hydroxyvitamin d",
        "serum vitamin d": "25-hydroxyvitamin d",
        "vitamin d (25-hydroxyvitamin d)": "25-hydroxyvitamin d",
        "vitamin d 25-hydroxy": "25-hydroxyvitamin d",
        "vitamin d, 25-hydroxy": "25-hydroxyvitamin d",
        "serum 25-hydroxyvitamin d": "25-hydroxyvitamin d",

        "vitamin b12": "vitamin b12",
        "vitamin b12 level": "vitamin b12",
        "serum vitamin b12": "vitamin b12",
        "vitamin b12 (cobalamin)": "vitamin b12",

        "thiamine": "thiamine",
        "thiamine level": "thiamine",
        "vitamin b1 (thiamine) level": "thiamine",
        "vitamin b1 level (thiamine)": "thiamine",

        "folate": "folate",
        "folic acid level": "folate",
        "serum folate": "folate",

        "vitamin b6 level": "vitamin b6",
        "serum vitamin b6 (pyridoxine)": "vitamin b6",

        "vitamin a level": "vitamin a",
        "serum vitamin a": "vitamin a",
        "serum retinol": "vitamin a",

        "vitamin c": "vitamin c",
        "ascorbic acid level": "vitamin c",
        "serum ascorbic acid level": "vitamin c",

        # ── Metabolic / other blood ──────────────────────────────
        "lactate": "lactate",
        "lactic acid": "lactate",
        "lactate level": "lactate",
        "serum lactate": "lactate",
        "blood lactate level": "lactate",
        "serum lactic acid": "lactate",

        "ammonia": "ammonia",
        "ammonia level": "ammonia",
        "ammonia lvl": "ammonia",
        "plasma ammonia": "ammonia",
        "serum ammonia": "ammonia",

        "amylase": "amylase",
        "amylase lvl": "amylase",

        "lipase": "lipase",
        "lipase lvl": "lipase",

        "procalcitonin": "procalcitonin",
        "serum procalcitonin": "procalcitonin",

        "crp": "crp",
        "c-reactive protein": "crp",
        "c-reactive protein (crp)": "crp",
        "high-sensitivity c-reactive protein (hs-crp)": "crp",

        "esr": "esr",
        "erythrocyte sedimentation rate": "esr",
        "erythrocyte sedimentation rate (esr)": "esr",

        "ldh": "ldh",
        "lactate dehydrogenase (ldh)": "ldh",

        "homocysteine": "homocysteine",
        "plasma homocysteine": "homocysteine",

        "prealbumin": "prealbumin",
        "prealbumin (transthyretin)": "prealbumin",
        "serum prealbumin": "prealbumin",

        "ceruloplasmin": "ceruloplasmin",
        "serum ceruloplasmin": "ceruloplasmin",

        "reticulocyte count": "reticulocyte count",
        "haptoglobin": "haptoglobin",
        "serum haptoglobin": "haptoglobin",
        "erythropoietin": "erythropoietin",
        "erythropoietin (epo)": "erythropoietin",
        "epo": "erythropoietin",

        "tryptase": "tryptase",
        "serum tryptase": "tryptase",
        "fibrin degradation products (fdps)": "fibrin degradation products",
        "d-dimer": "d-dimer",

        "carboxyhemoglobin": "carboxyhemoglobin",
        "carboxyhemoglobin (cohb)": "carboxyhemoglobin",
        "methemoglobin": "methemoglobin",
        "methemoglobin level": "methemoglobin",

        "beta-hydroxybutyrate": "beta-hydroxybutyrate",
        "blood ketones": "beta-hydroxybutyrate",
        "ketones": "ketones",
        "serum ketones": "ketones",

        "osmolality": "osmolality",
        "serum osmolal gap": "serum osmolal gap",
        "osmolal gap": "serum osmolal gap",

        "plasma metanephrines": "plasma metanephrines",
        "plasma free metanephrines": "plasma metanephrines",
        "urinary metanephrines": "urinary metanephrines",

        "haptoglobin": "haptoglobin",

        # ── ABG ──────────────────────────────────────────────────
        "arterial blood gas": "arterial blood gas",
        "arterial blood gas (abg)": "arterial blood gas",
        "abg": "arterial blood gas",
        "blood gas analysis": "arterial blood gas",
        "blood gas analysis (abg)": "arterial blood gas",

        "ph": "ph",
        "blood ph": "ph",
        "serum ph": "ph",

        "pco2": "pco2",
        #"paco2": "pco2",
        "pc02": "pco2",

        "po2": "po2",
        "pao2": "po2",

        "fio2": "fio2",
        "base excess": "base excess",

        "oxygen saturation": "oxygen saturation",
        "o2 sat": "oxygen saturation",
        "o2 sat calc": "oxygen saturation",
        "spo2": "oxygen saturation",
        "pulse oximetry": "oxygen saturation",
        "oxygen saturation (spo2)": "oxygen saturation",
        "pulse oximetry (spo2)": "oxygen saturation",
        "oxygen via nasal cannula": "oxygen saturation",

        # ── Vital signs ──────────────────────────────────────────
        "heart rate": "heart rate",
        "hr": "heart rate",
        "heart rate (hr)": "heart rate",
        "vital sign: heart rate": "heart rate",
        "maternal heart rate": "heart rate",
        "fetal heart rate": "fetal heart rate",

        "respiratory rate": "respiratory rate",
        "rr": "respiratory rate",

        "temperature": "temperature",
        "temp": "temperature",
        "patient temp": "temperature",
        "vital sign: temperature": "temperature",
        "maternal temperature": "temperature",
        "hot": "temperature",

        "blood pressure": "blood pressure",
        "systolic bp": "systolic bp",
        "systolic blood pressure measurement": "systolic bp",
        "sbp": "systolic bp",
        "bp systolic": "systolic bp",
        "blood pressure systolic": "systolic bp",
        "blood pressure systolic (sbp)": "systolic bp",
        "systolic blood pressure": "systolic bp",
        "maternal blood pressure systolic": "systolic bp",

        "diastolic bp": "diastolic bp",
        "diastolic blood pressure measurement": "diastolic bp",
        "dbp": "diastolic bp",
        "bp diastolic": "diastolic bp",
        "blood pressure diastolic": "diastolic bp",
        "blood pressure diastolic (dbp)": "diastolic bp",
        "diastolic blood pressure": "diastolic bp",

        "bmi": "bmi",
        "body mass index": "bmi",
        "body mass index (bmi)": "bmi",
        "weight": "weight",
        "height": "height",
        "head circumference": "head circumference",
        "waist circumference": "waist circumference",
        "pain score": "pain score",

        # ── CSF ──────────────────────────────────────────────────
        "csf analysis": "csf analysis",
        "lumbar puncture": "csf analysis / lumbar puncture",
        "csf protein": "csf protein",
        "csf glucose": "csf glucose",
        "csf culture": "csf culture",
        "csf gram stain": "csf gram stain",
        "csf vdrl": "csf vdrl",
        "csf cell count": "csf cell count",
        "csf white blood cell count": "csf cell count",
        "csf viral pcr": "csf viral pcr",
        "csf fta-abs": "csf fta-abs",

        # ── Cultures / microbiology ──────────────────────────────
        "blood culture": "blood culture",
        "blood cultures": "blood culture",
        "urine culture and sensitivity": "urine culture and sensitivity",
        "wound culture": "wound culture",
        "sputum culture": "sputum culture",
        "throat culture": "throat culture",
        "nasal culture": "nasal culture",
        "stool culture": "stool culture",
        "viral panel cultures": "viral culture",
        "gram stain": "gram stain",
        "koh preparation": "koh preparation",
        "wet mount": "wet mount",
        "peripheral blood smear": "peripheral blood smear",

        # ── Serology / immunology ────────────────────────────────
        "rpr": "rpr",
        "rapid plasma reagin (rpr)": "rpr",
        "vdrl": "vdrl",
        "ana": "ana",
        "antinuclear antibody (ana)": "ana",
        "rheumatoid factor": "rheumatoid factor",
        "rheumatoid factor (rf)": "rheumatoid factor",
        "rf": "rheumatoid factor",
        "hla-b27": "hla-b27",
        "anti-ccp antibody": "anti-ccp antibody",
        "anca": "anca",
        "anti-gbm antibody": "anti-gbm antibody",
        "complement c3": "complement c3",
        "c3 complement level": "complement c3",
        "complement c4": "complement c4",
        "c4 level": "complement c4",
        "anti-dsdna": "anti-dsdna",
        "anticardiolipin antibodies (igg, igm)": "anticardiolipin antibodies",
        "antiphospholipid antibodies (apl)": "antiphospholipid antibodies",
        "immunoglobulin levels (igg, iga, igm, ige)": "immunoglobulin levels",
        "ige level": "ige level",
        "ige levels": "ige level",
        "total serum ige": "ige level",
        "serum ige": "ige level",
        "total immunoglobulin e (ige)": "ige level",

        # ── Infectious disease ───────────────────────────────────
        "hiv viral load": "hiv viral load",
        "hiv rna (viral load)": "hiv viral load",
        "hiv rna level (viral load)": "hiv viral load",
        "hiv test": "hiv test",
        "cd4+ t-cell count": "cd4 count",
        "cd4 count": "cd4 count",
        "cd4+ t-lymphocyte count": "cd4 count",
        "hepatitis b surface antigen (hbsag)": "hepatitis b surface antigen",
        "hcv rna": "hcv rna",
        "cmv pcr": "cmv pcr",
        "cmv dna pcr": "cmv pcr",
        "ebv viral load": "ebv viral load",
        "hsv pcr": "hsv pcr",
        "influenza a ag": "influenza antigen",
        "influenza b ag": "influenza antigen",
        "influenza s ag": "influenza antigen",
        "influenza a/b pcr": "influenza test",
        "influenza a/b rapid test": "influenza test",
        "influenza a/b test": "influenza test",
        "rsv ag": "rsv test",
        "sars ag": "sars-cov-2 test",
        "sars-cov-2 pcr test": "sars-cov-2 test",
        "sars-cov-2 rt-pcr": "sars-cov-2 test",
        "rapid strep scr": "rapid strep test",
        "rapid strep test": "rapid strep test",
        "legionella urinary antigen test": "legionella urinary antigen",
        "cryptococcal antigen (crag)": "cryptococcal antigen",
        "galactomannan antigen": "galactomannan",
        "beta-d-glucan": "beta-d-glucan",
        "lyme disease serology": "lyme disease serology",
        "dengue ns1 antigen": "dengue serology",
        "dengue igm antibodies": "dengue serology",
        "malaria smear (thick and thin)": "malaria smear",
        "widal test": "widal test",
        "monospot test (heterophile antibody test)": "monospot test",
        "hepatitis a igm antibody": "hepatitis a igm",
        "blood type and screen": "blood type and screen",
        "direct antiglobulin test (dat)": "direct coombs test",
        "direct coombs test (direct antiglobulin test)": "direct coombs test",
        "indirect coombs test (indirect antiglobulin test)": "indirect coombs test",
        "kleihauer-betke test": "kleihauer-betke test",
        "kleihauer-betke (kb) test": "kleihauer-betke test",

        # ── Tumor markers ────────────────────────────────────────
        "psa": "psa",
        "prostate-specific antigen": "psa",
        "prostate-specific antigen (psa)": "psa",
        "ca-125": "ca-125",
        "cancer antigen 125 (ca-125)": "ca-125",
        "cea": "cea",
        "carcinoembryonic antigen (cea)": "cea",
        "ca 19-9": "ca 19-9",
        "afp": "afp",
        "alpha-fetoprotein (afp)": "afp",
        "beta-2 microglobulin": "beta-2 microglobulin",
        "ca 15-3": "ca 15-3",
        "chromogranin a": "chromogranin a",

        # ── Protein / SPEP ───────────────────────────────────────
        "serum protein electrophoresis": "serum protein electrophoresis",
        "serum protein electrophoresis (spep)": "serum protein electrophoresis",
        "spep": "serum protein electrophoresis",
        "serum free light chains": "serum free light chains",
        "immunofixation electrophoresis (ife)": "immunofixation electrophoresis",
        "ife": "immunofixation electrophoresis",

        # ── Drug levels ──────────────────────────────────────────
        "acetaminophen level": "acetaminophen level",
        "serum acetaminophen level": "acetaminophen level",
        "acetaminph lvl": "acetaminophen level",
        "digoxin level": "digoxin level",
        "lithium level": "lithium level",
        "phenytoin level": "phenytoin level",
        "valproic acid level": "valproic acid level",
        "carbamazepine level": "carbamazepine level",
        "tacrolimus trough level": "tacrolimus level",
        "tacrolimus level": "tacrolimus level",
        "ethanol level": "ethanol level",
        "ethanol lvl": "ethanol level",
        "ethanol": "ethanol level",
        "blood alcohol level": "ethanol level",
        "blood alcohol level (bal)": "ethanol level",
        "salicylate level": "salicylate level",
        "salicylate": "salicylate level",
        "methanol level": "methanol level",
        "barbiturate level": "barbiturate level",
        "benzodiazepine level": "benzodiazepine level",
        "lead level": "lead level",
        "blood lead level": "lead level",
        "toxicology screen": "toxicology screen",
        "vancomycin level": "vancomycin level",
        "aminoglycoside level": "aminoglycoside level",

        # ── Panels ───────────────────────────────────────────────
        "lipid panel": "lipid panel",
        "iron studies": "iron studies",
        "thyroid function tests": "thyroid function tests",
        "renal function tests": "renal function tests",
        "coagulation panel": "coagulation panel",
        "electrolytes": "electrolytes",
        "serum electrolytes": "electrolytes",

        # ── ECG / cardiac imaging ────────────────────────────────
        "ecg": "ecg",
        "ekg": "ecg",
        "ecg/ekg": "ecg",
        "electrocardiogram": "ecg",
        "electrocardiogram (ecg)": "ecg",
        "12-lead electrocardiogram (ecg/ekg)": "ecg",
        "echocardiogram": "echocardiogram",
        "echo": "echocardiogram",
        "transthoracic echocardiogram (tte)": "echocardiogram",
        "tee": "echocardiogram",
        "cardiac catheterization": "cardiac cath",
        "cardiac cath": "cardiac cath",
        "right heart catheterization": "right heart catheterization",

        # ── Neuro ────────────────────────────────────────────────
        "eeg": "eeg",
        "electroencephalogram (eeg)": "eeg",
        "emg": "emg",
        "emg/ncs": "emg",
        "electromyography (emg)": "emg",
        "nerve conduction studies": "nerve conduction studies",
        "nerve conduction studies (ncs)": "nerve conduction studies",
        "glasgow coma scale": "glasgow coma scale",
        "glasgow coma scale (gcs)": "glasgow coma scale",
        "nihss": "nihss",
        "nih stroke scale (nihss)": "nihss",
        "national institutes of health stroke scale (nihss)": "nihss",
        "intracranial pressure": "intracranial pressure",
        "intracranial pressure (icp)": "intracranial pressure",
        "icp monitoring": "intracranial pressure",
        "central venous pressure": "central venous pressure",
        "central venous pressure (cvp)": "central venous pressure",

        # ── Imaging ──────────────────────────────────────────────
        "chest x-ray": "chest x-ray",
        "cxr": "chest x-ray",
        "ct": "ct scan",
        "ct head": "ct head",
        "head ct": "ct head",
        "head ct scan": "ct head",
        "non-contrast head ct": "ct head",
        "cta": "ct angiography",
        "cta chest": "ct angiography chest",
        "cta chest w/ contrast": "ct angiography chest",
        "cta head": "ct angiography head",
        "ct abdomen": "ct abdomen",
        "ct abdomen and pelvis with contrast": "ct abdomen/pelvis",
        "ct abdomen and pelvis w/ contrast": "ct abdomen/pelvis",
        "ct abdomen/pelvis": "ct abdomen/pelvis",
        "ct pelvis": "ct pelvis",
        "ct maxillofacial": "ct maxillofacial",
        "mri": "mri",
        "mri brain": "mri brain",
        "brain mri": "mri brain",
        "mri spine cervical": "mri cervical spine",
        "mri spine thoracic": "mri thoracic spine",
        "abdominal ultrasound": "abdominal ultrasound",
        "ruq ultrasound": "abdominal ultrasound",
        "pelvic ultrasound": "pelvic ultrasound",
        "renal ultrasound": "renal ultrasound",
        "kidney ultrasound": "renal ultrasound",
        "scrotal ultrasound": "scrotal ultrasound",
        "carotid ultrasound": "carotid ultrasound",
        "doppler ultrasound": "doppler ultrasound",
        "ultrasound": "ultrasound",
        "x-ray": "x-ray",
        "xr pelvis": "x-ray pelvis",
        "xr pelvis 1 view": "x-ray pelvis",
        "adrenal washout ct": "adrenal washout ct",
        "bone scan": "bone scan",
        "pet/ct scan": "pet/ct scan",
        "mammography": "mammography",
        "mammogram": "mammography",
        "bone marrow biopsy": "bone marrow biopsy",
        "dexa scan": "dexa scan",
        "bone mineral density (bmd) scan (dexa)": "dexa scan",
        "pcr": "pcr",

        # ── Pulm ─────────────────────────────────────────────────
        "pulmonary function tests": "pulmonary function tests",
        "pulmonary function tests (pfts)": "pulmonary function tests",
        "spirometry": "pulmonary function tests",
        "peak expiratory flow (pef)": "peak expiratory flow",
        "dlco": "dlco",
        "fvc": "fvc",

        # ── GI / procedures ─────────────────────────────────────
        "colonoscopy": "colonoscopy",
        "upper endoscopy": "upper endoscopy",
        "upper endoscopy (egd)": "upper endoscopy",
        "esophagogastroduodenoscopy (egd)": "upper endoscopy",
        "bronchoscopy": "bronchoscopy",
        "cystoscopy": "cystoscopy",
        "liver biopsy": "liver biopsy",
        "kidney biopsy": "kidney biopsy",
        "polysomnography": "polysomnography",
        "sleep study (polysomnography)": "polysomnography",
        "helicobacter pylori testing": "helicobacter pylori testing",
        "celiac disease serology": "celiac disease serology",
        "fecal calprotectin": "fecal calprotectin",
        "fecal occult blood test (fobt)": "fecal occult blood test",
        "stool occult blood test": "fecal occult blood test",
        "stool ova and parasites": "stool ova and parasites",
        "stool for ova and parasites": "stool ova and parasites",
        "stool for ova and parasites (o&p)": "stool ova and parasites",
        "stool c. difficile toxin": "stool c. difficile toxin",
        "stool for c. difficile toxin": "stool c. difficile toxin",
        "fecal elastase-1": "fecal elastase-1",

        # ── OB/GYN ───────────────────────────────────────────────
        "non-stress test": "non-stress test",
        "non-stress test (nst)": "non-stress test",
        "nonstress test (nst)": "non-stress test",
        "biophysical profile": "biophysical profile",
        "biophysical profile (bpp)": "biophysical profile",
        "fetal heart rate monitoring": "fetal heart rate monitoring",
        "cardiotocography": "cardiotocography",
        "cardiotocography (ctg)": "cardiotocography",
        "group b strep screening": "group b strep screening",
        "group b streptococcus (gbs) culture": "group b strep screening",
        "cervical length": "cervical length",
        "fetal fibronectin": "fetal fibronectin",
        "amniocentesis": "amniocentesis",
        "nipt": "nipt",
        "non-invasive prenatal testing (nipt)": "nipt",
        "pap smear": "pap smear",
        "hcg qual u": "pregnancy test / hcg",
        "pregnancy test": "pregnancy test / hcg",
        "quantitative beta-hcg": "pregnancy test / hcg",
        "quantitative hcg (beta-hcg)": "pregnancy test / hcg",
        "quantitative hcg": "pregnancy test / hcg",

        # ── Eye ──────────────────────────────────────────────────
        "intraocular pressure": "intraocular pressure",
        "intraocular pressure (iop)": "intraocular pressure",
        "iop": "intraocular pressure",
        "tonometry": "intraocular pressure",
        "visual acuity": "visual acuity",
        "visual field testing": "visual field testing",
        "slit-lamp examination": "slit-lamp examination",
        "slit lamp examination": "slit-lamp examination",
        "oct": "oct",
        "optical coherence tomography (oct)": "oct",
        "fluorescein angiography": "fluorescein angiography",
        "fundus examination": "fundus examination",
        "fundoscopic examination": "fundus examination",
        "fundoscopic exam": "fundus examination",
        "ophthalmoscopy": "fundus examination",
        "tear break-up time": "tear break-up time",
        "tear break-up time (tbut)": "tear break-up time",
        "gonioscopy": "gonioscopy",
        "schirmer's test": "schirmer's test",

        # ── Ear ──────────────────────────────────────────────────
        "audiometry": "audiometry",
        "pure tone audiometry": "audiometry",
        "tympanometry": "tympanometry",
        "otoacoustic emissions (oae)": "otoacoustic emissions",
        "auditory brainstem response (abr)": "auditory brainstem response",
        "baer": "auditory brainstem response",

        # ── Miscellaneous clinical ────────────────────────────────
        "semen analysis": "semen analysis",
        "bone age x-ray": "bone age x-ray",
        "bone age": "bone age x-ray",
        "ankle-brachial index": "ankle-brachial index",
        "ankle-brachial index (abi)": "ankle-brachial index",
        "sweat chloride test": "sweat chloride test",
        "sweat test": "sweat chloride test",
        "ciwa-ar (clinical institute withdrawal assessment for alcohol - revised)": "ciwa-ar score",
        "ciwa-ar scale": "ciwa-ar score",
        "peripheral blood smear": "peripheral blood smear",
        "blood type and screen": "blood type and screen",
        "direct coombs test": "direct coombs test",
        "indirect coombs test": "indirect coombs test",
        "kleihauer-betke test": "kleihauer-betke test",
        "glucose tolerance test": "glucose tolerance test",
        "meld score": "meld score",
        "bun/creatinine ratio": "bun/creatinine ratio",
        "genetic testing / karyotype": "genetic testing / karyotype",
        "genetic testing": "genetic testing / karyotype",
        "karyotype": "genetic testing / karyotype",
        "chromosomal microarray analysis (cma)": "chromosomal microarray analysis",
        "estimated average glucose": "estimated average glucose",
        "eag": "estimated average glucose",
        "estimated average glucose (eag)": "estimated average glucose",
        "hepatitis steatosis": "hepatic steatosis",
        "age": "age",

    # ════════════════════════════════════════════════════════
    # NEW ENTRIES — added from unique_labs.xlsx
    # Rules: CT variants→"ct", MRI→"mri", Ultrasound→"ultrasound",
    #         Biopsy→"biopsy", X-ray→"x-ray" (and more below)
    # ════════════════════════════════════════════════════════

    # ── CT scan variants ────────────────────────────────
    "abdominal ct scan": "ct",
    "angiography (mra/cta)": "ct",
    "cardiac computed tomography": "ct",
    "chest computerized tomography (ct) scan": "ct",
    "chest ct angiography": "ct",
    "computed tomography": "ct",
    "computed tomography (ct) scan": "ct",
    "computed tomography angiography": "ct",
    "ct and mri scans": "ct",
    "ct angiography (cta)": "ct",
    "ct arthrography": "ct",
    "ct chest": "ct",
    "ct coronary angiogram": "ct",
    "ct exan": "ct",
    "ct pulmonary angiography (ctpa)": "ct",
    "ct scan": "ct",
    "ct scan abdomen and pelvis": "ct",
    "ct scan nasal cultures": "ct",
    "ct scan of the heart": "ct",
    "ct scan or mri": "ct",
    "ct scans": "ct",
    "ct scans / angiograms": "ct",
    "ct scans and mri scans of the abdomen and pelvis": "ct",
    "ct-guided needle biopsy": "ct",
    "ct/mr venography": "ct",
    "digital subtraction angiography": "ct",
    "dual-energy computed tomography (dect)": "ct",
    "high-resolution ct": "ct",
    "mra and cta": "ct",
    "multidetector ct scan": "ct",
    "non-contrast helical ct scan of abdomen and pelvis": "ct",
    "oct angiography": "ct",
    "pet-ct scan": "ct",
    "ultrasound and ct scans": "ct",
    "virtual colonoscopy (ct colonography)": "ct",

    # ── MRI variants ────────────────────────────────────
    "also called a cardiac mri": "mri",
    "brain mri or ct": "mri",
    "breast mri": "mri",
    "cardiac magnetic resonance": "mri",
    "cardiac magnetic resonance imaging": "mri",
    "fetal mri": "mri",
    "functional mri (fmri)": "mri",
    "heart mri scan": "mri",
    "magnetic resonance angiography": "mri",
    "magnetic resonance angiography (mra)": "mri",
    "magnetic resonance imaging": "mri",
    "magnetic resonance imaging (mri)": "mri",
    "mr angiography (mra)": "mri",
    "mr cholangiopancreatography (mrcp)": "mri",
    "mra": "mri",
    "mrcp": "mri",
    "mri and mrcp": "mri",
    "mri diagnostic ultrasound": "mri",
    "mri of the brain": "mri",
    "mri scan": "mri",
    "mri scans": "mri",
    "mri scans of the head": "mri",
    "mri/mrcp": "mri",
    "pituitary mri": "mri",
    "ultrasound and cine-mri": "mri",

    # ── Ultrasound variants ─────────────────────────────
    "abdominal ultrasound (us)": "ultrasound",
    "b-scan ultrasound": "ultrasound",
    "breast ultrasound": "ultrasound",
    "carotid ultrasound / doppler": "ultrasound",
    "chest ultrasound": "ultrasound",
    "color doppler": "ultrasound",
    "cranial ultrasound": "ultrasound",
    "diaphragmatic ultrasound": "ultrasound",
    "duplex ultrasound": "ultrasound",
    "endobronchial ultrasound": "ultrasound",
    "endoscopic ultrasonography (eus)": "ultrasound",
    "endoscopic ultrasound": "ultrasound",
    "eus": "ultrasound",
    "fetal ultrasound": "ultrasound",
    "focused assessment with sonography": "ultrasound",
    "handheld doppler": "ultrasound",
    "lung ultrasound": "ultrasound",
    "mid-pregnancy ultrasound": "ultrasound",
    "nerve ultrasound": "ultrasound",
    "saline infusion sonography": "ultrasound",
    "second-trimester ultrasound": "ultrasound",
    "stress sonography": "ultrasound",
    "testicular ultrasound": "ultrasound",
    "thyroid ultrasound": "ultrasound",
    "ultrasonography": "ultrasound",
    "ultrasound bladder": "ultrasound",
    "ultrasound kidney and bladder": "ultrasound",
    "ultrasound of prostate": "ultrasound",
    "vascular ultrasound": "ultrasound",
    "venous doppler ultrasound": "ultrasound",

    # ── Biopsy variants ─────────────────────────────────
    "biopsies": "biopsy",
    "biopsy": "biopsy",
    "biopsy (core needle": "biopsy",
    "biopsy and tissue analysis": "biopsy",
    "bone marrow analysis": "biopsy",
    "bone marrow examination": "biopsy",
    "breast biopsy": "biopsy",
    "capsular & tissue analysis": "biopsy",
    "colon biopsy": "biopsy",
    "endometrial biopsy": "biopsy",
    "esophageal biopsy": "biopsy",
    "fine needle aspiration": "biopsy",
    "histopathological biopsy of colon": "biopsy",
    "histopathology": "biopsy",
    "immunohistochemistry": "biopsy",
    "immunohistochemistry (ihc)": "biopsy",
    "kidney tumor biopsy": "biopsy",
    "lip or tissue biopsy": "biopsy",
    "lymph node biopsy": "biopsy",
    "mucosal biopsy": "biopsy",
    "muscle biopsy": "biopsy",
    "myocardial biopsy": "biopsy",
    "nerve / skin biopsy": "biopsy",
    "peritoneal biopsy & histology": "biopsy",
    "pituitary biopsy & histology": "biopsy",
    "pleural biopsy": "biopsy",
    "prostate biopsy": "biopsy",
    "punch or excisional biopsy of vulva": "biopsy",
    "skin biopsy": "biopsy",
    "thyroid fine-needle aspiration (fna) biopsy": "biopsy",
    "tissue sample": "biopsy",
    "tissue sampling": "biopsy",

    # ── X-ray variants ──────────────────────────────────
    "abdominal or chest x-ray": "x-ray",
    "abdominal x-ray": "x-ray",
    "bladder (kub) x-ray": "x-ray",
    "dental x-rays": "x-ray",
    "kub x-ray": "x-ray",
    "plain abdominal x-ray": "x-ray",
    "plain radiography (x-ray)": "x-ray",
    "upper gi series x-ray": "x-ray",

    # ── PET scan variants ───────────────────────────────
    "fdg-pet scan": "pet scan",
    "pet": "pet scan",
    "pet and spect scans": "pet scan",
    "pet scan": "pet scan",
    "pet scans": "pet scan",
    "positron emission tomography": "pet scan",
    "positron emission tomography (pet) scans": "pet scan",

    # ── SPECT variants ──────────────────────────────────
    "datscan": "spect",
    "single-photon emission computerized tomography (spect) scan called a dopamine transporter (dat) scan": "spect",
    "spect": "spect",

    # ── Endoscopy variants ──────────────────────────────
    "anoscopy": "endoscopy",
    "colonoscopy or sigmoidoscopy": "endoscopy",
    "cystourethroscopy": "endoscopy",
    "endoscopic retrograde cholangiopancreatography": "endoscopy",
    "endoscopic retrograde cholangiopancreatography (ercp)": "endoscopy",
    "endoscopy": "endoscopy",
    "endoscopy procedures": "endoscopy",
    "ercp": "endoscopy",
    "esophagogastroduodenoscopy": "endoscopy",
    "fiber-optic nasal endoscopy": "endoscopy",
    "fiberoptic endoscopic evaluation of swallowing": "endoscopy",
    "flexible colonoscopy": "endoscopy",
    "flexible sigmoidoscopy": "endoscopy",
    "hysteroscopy": "endoscopy",
    "mediastinoscopy": "endoscopy",
    "nasal endoscopy": "endoscopy",
    "proctoscopy": "endoscopy",
    "thoracoscopy (vats)": "endoscopy",
    "upper gi endoscopy": "endoscopy",
    "video capsule endoscopy": "endoscopy",
    "voiding cystourethrogram (vcug)": "endoscopy",

    # ── ECG variants ────────────────────────────────────
    "electrocardiogram (ecg/ekg)": "ecg",

    # ── Echocardiogram variants ─────────────────────────
    "contrast echocardiography": "echocardiogram",
    "echocardiography": "echocardiogram",
    "fetal echocardiogram": "echocardiogram",
    "transesophageal echocardiogram": "echocardiogram",
    "transthoracic echocardiography": "echocardiogram",

    # ── EMG / NCS variants ──────────────────────────────
    "electromyography": "emg/nerve conduction study",
    "electromyography (emg) and nerve conduction studies (ncs": "emg/nerve conduction study",
    "emg/ncv (electromyography & nerve conduction velocity)": "emg/nerve conduction study",
    "evoked potential tests": "emg/nerve conduction study",
    "nerve conduction velocity (ncv) studies": "emg/nerve conduction study",
    "somatosensory evoked potentials": "emg/nerve conduction study",

    # ── EEG variants ────────────────────────────────────
    "electrical source imaging (esi)": "eeg",
    "electroencephalogram": "eeg",
    "electroencephalography": "eeg",
    "high-density eeg": "eeg",
    "magnetoencephalography": "eeg",
    "qeeg (quantitative eeg)": "eeg",

    # ── Pulmonary function ──────────────────────────────
    "bronchodilator reversibility test": "pulmonary function tests",
    "diffusion tests": "pulmonary function tests",
    "fractional exhaled nitric oxide test": "pulmonary function tests",
    "lung volume test": "pulmonary function tests",
    "lung volume test & diffusion capacity": "pulmonary function tests",
    "peak flow measurement": "pulmonary function tests",
    "pulmonary function test": "pulmonary function tests",
    "spirometry and diffusion capacity": "pulmonary function tests",

    # ── PCR / molecular ─────────────────────────────────
    "hcv rna test (pcr)": "pcr",
    "molecular tests": "pcr",
    "molecular tests (e.g.": "pcr",
    "molecular tests (naat/pcr)": "pcr",
    "pcr (polymerase chain reaction)": "pcr",
    "rt-pcr)": "pcr",

    # ── Culture variants ────────────────────────────────
    "culture": "culture",
    "fluid culture": "culture",
    "skin/wound culture": "culture",

    # ── Nuclear medicine ────────────────────────────────
    "gastric emptying scintigraphy": "nuclear medicine scan",
    "hepatobiliary iminodiacetic acid scan": "nuclear medicine scan",
    "lymphoscintigraphy": "nuclear medicine scan",
    "nuclear medicine scans": "nuclear medicine scan",
    "ventilation-perfusion scan": "nuclear medicine scan",

    # ── Lumbar puncture / CSF ───────────────────────────
    "cerebrospinal fluid (csf) analysis": "lumbar puncture",
    "csf test": "lumbar puncture",

    # ── Genetic testing ─────────────────────────────────
    "chromosomal microarray (acgh)": "genetic testing",
    "dna analysis": "genetic testing",
    "gene-targeted duplication/deletion analysis": "genetic testing",
    "hemoglobin electrophoresis/genetic testing": "genetic testing",
    "targeted gene panels": "genetic testing",
    "whole-exome sequencing (wes) or whole-genome sequencing (wgs)": "genetic testing",

    # ── Angiography ─────────────────────────────────────
    "angiogram": "angiography",
    "angiography": "angiography",
    "cardiac catheterization and angiogram": "cardiac cath",
    "cerebral angiography": "angiography",
    "coronary angiogram": "angiography",
    "invasive coronary angiography": "angiography",

    # ── Stress test ─────────────────────────────────────
    "exercise stress tests": "stress test",
    "stress test": "stress test",
    "stress testing": "stress test",

    # ── Polysomnography / sleep ─────────────────────────
    "actigraphy": "polysomnography",
    "multiple sleep latency test": "polysomnography",
    "sleep logs and actigraphy": "polysomnography",

    # ── Audiometry / ENT ────────────────────────────────
    "acoustic rhinometry": "nasal examination",
    "anterior rhinoscopy": "nasal examination",
    "audiography": "audiometry",
    "audiometric testing": "audiometry",
    "auditory brainstem response": "auditory brainstem response",
    "bone conduction testing": "bone conduction testing",
    "nasal smear": "nasal examination",
    "otoacoustic emissions": "otoacoustic emissions",
    "otoscopy": "otoscopy",
    "pneumatic otoscopy": "otoscopy",
    "pure-tone audiometry": "audiometry",
    "rhinomanometry": "nasal examination",
    "speech audiometry": "audiometry",
    "tuning fork tests": "tuning fork tests",

    # ── Eye tests ───────────────────────────────────────
    "dacryocystography": "dacryocystography",
    "electric pulp testing": "electric pulp testing",
    "electroretinography": "electroretinography",
    "fluorescein dye disappearance test (fddt)": "fluorescein dye disappearance test",
    "fundus autofluorescence": "fundoscopy",
    "fundus photography": "fundoscopy",
    "oct (optical coherence tomography)": "optical coherence tomography",
    "ophthalmoscopy (optic nerve exam)": "fundoscopy",
    "optical coherence tomography": "optical coherence tomography",
    "pachymetry": "pachymetry",
    "perimetry (visual field test)": "visual field test",
    "retinal imaging": "retinal imaging",
    "tonometry (eye pressure test)": "intraocular pressure",

    # ── Inflammatory / CBC ──────────────────────────────
    "band count calculation": "band count",
    "hemoglobin (hb) and hematocrit (hct)": "hemoglobin and hematocrit",
    "inflammatory markers": "inflammatory markers",
    "mcv (mean corpuscular volume) & mch": "mcv/mch",
    "peripheral smear": "peripheral blood smear",

    # ── Chemistry / metabolic ───────────────────────────
    "ammonia levels": "ammonia",
    "basic metabolic panel (bmp)": "basic metabolic panel",
    "blood uric acid test": "uric acid",
    "comprehensive metabolic panel (cmp)": "comprehensive metabolic panel",
    "egfr check": "egfr",
    "electrolyte panel or basic metabolic panel": "basic metabolic panel",
    "fractional excretion of sodium (fena)": "fractional excretion of sodium",
    "fractionated bilirubin": "bilirubin",
    "peth (phosphatidylethanol)": "phosphate",
    "plasma ammonia measurement": "ammonia",
    "proteinuria": "urine protein",
    "total protein and albumin tests": "albumin",
    "urea & creatinine excretion": "creatinine",
    "urine sodium & potassium": "urine electrolytes",

    # ── Liver / GI tests ────────────────────────────────
    "barium swallow": "barium swallow study",
    "barium swallow and stool test": "barium swallow study",
    "barium swallow study": "barium swallow study",
    "contrast enema": "contrast enema",
    "contrast enema (fistulogram)": "contrast enema",
    "diagnostic peritoneal lavage": "diagnostic peritoneal lavage",
    "direct visualization": "direct visualization",
    "esophageal manometry": "manometry",
    "esophagography": "esophagography",
    "fecal immunochemical test": "stool test",
    "fibroscan (elastography)": "elastography",
    "fluoroscopy": "fluoroscopy",
    "functional lumen imaging probe": "functional lumen imaging probe",
    "gamma-glutamyl transferase": "ggt",
    "high-resolution manometry (hrm)": "manometry",
    "intraoperative cholangiography": "intraoperative cholangiography",
    "paracentesis": "paracentesis",
    "peritoneal fluid analysis": "peritoneal fluid analysis",
    "radiologic fluoroscopy": "fluoroscopy",
    "radiopaque marker study": "radiopaque marker study",
    "salivary diagnostics": "salivary diagnostics",
    "sputum analysis": "sputum analysis",
    "sputum or gastric aspirate microscopy": "sputum analysis",
    "sputum test": "sputum analysis",
    "sputum tests": "sputum analysis",
    "stool dna test": "stool test",
    "stool test": "stool test",
    "stool tests": "stool test",
    "transient elastography": "elastography",
    "wireless motility capsule": "wireless motility capsule",

    # ── Cardiac tests ───────────────────────────────────
    "active stand test (nasa lean test)": "tilt table test",
    "b-type natriuretic peptide (bnp) blood test": "bnp",
    "coronary calcium score": "coronary calcium score",
    "head-up tilt table test": "tilt table test",
    "holter monitor": "holter monitor",

    # ── Endocrine / hormones ────────────────────────────
    "acth stimulation test": "acth stimulation test",
    "adrenal venous sampling (avs)": "adrenal venous sampling",
    "aldosterone-renin ratio": "aldosterone-to-renin ratio",
    "androgens": "androgens",
    "captopril challenge test (cct)": "captopril challenge test",
    "dexamethasone suppression test": "dexamethasone suppression test",
    "fludrocortisone suppression test (fst)": "fludrocortisone suppression test",
    "insulin tolerance test (itt)": "insulin tolerance test",
    "thyroid peroxidase (tpo) antibodies": "thyroid peroxidase antibodies",
    "total or free t₃ (triiodothyronine) test": "free t3",
    "tsh (thyroid-stimulating hormone) test": "tsh",
    "t₄ (thyroxine) test": "free t4",

    # ── Diabetes tests ──────────────────────────────────
    "fasting / caloric deprivation test": "fasting test",
    "fasting plasma glucose (fpg)": "fasting plasma glucose",
    "oral glucose tolerance test (ogtt)": "oral glucose tolerance test",

    # ── Lipids / vitamins ───────────────────────────────
    "lipid profile": "lipid panel",
    "transferrin saturation (tsat)": "iron studies",

    # ── Oncology markers ────────────────────────────────
    "ca 19-9 and ce": "ca 19-9",
    "ca-125 for females": "ca-125",
    "cea and he4": "cea",
    "chromogranin a (cga)": "chromogranin a",
    "lysosphingolipid panels": "lysosphingolipid panel",
    "prostate-specific membrane antigen (psma)": "psma",
    "tumor marker (ca 19-9)": "ca 19-9",
    "tumor marker ca-125": "ca-125",

    # ── Infectious disease ──────────────────────────────
    "anti-hbc or hbcab (total hepatitis b core antibody)": "hepatitis b core antibody",
    "anti-hbs or hbsab (hepatitis b surface antibody)": "hepatitis b surface antibody",
    "antigen tests": "antigen tests",
    "confirmatory test (hcv rna test)": "hepatitis c rna",
    "covid-19 antigen tests": "covid-19 test",
    "gdh antigen tests": "gdh antigen",
    "genotype testing for hcv": "hepatitis c rna",
    "hbsag (hepatitis b surface antigen)": "hepatitis b surface antigen",
    "hcv antibody test": "hepatitis c antibody",
    "igm anti-hbc (igm core antibody)": "hepatitis b core antibody",
    "initial screening (hcv antibody test)": "hepatitis c antibody",
    "rapid influenza test": "influenza rapid test",
    "sars-cov-2": "covid-19 test",
    "toxin tests (eia)": "toxin test",

    # ── Immunology / serology ───────────────────────────
    "activated partial thromboplastin time": "aptt",
    "anti-ccp (anti-cyclic citrullinated peptide)": "anti-ccp antibody",
    "antiphospholipid syndrome (aps) tests": "antiphospholipid antibodies",
    "cdt (carbohydrate-deficient transferrin)": "cdt",
    "etg (ethyl glucuronide) & ets (ethyl sulfate)": "alcohol biomarkers",
    "factor ix for hemophilia b": "coagulation factors",
    "factor v leiden (fvl) and prothrombin g20210a": "coagulation factors",
    "factor viii for hemophilia a": "coagulation factors",
    "ige blood tests": "ige blood tests",
    "natural coagulation inhibitors": "coagulation factors",
    "paraneoplastic antibody panels": "paraneoplastic antibody panel",
    "qsart": "autonomic testing",
    "rheumatoid factor test (rf test)": "rheumatoid factor",
    "thrombin time (tt) / fibrinogen test": "thrombin time/fibrinogen",

    # ── OB/GYN / reproductive ───────────────────────────
    "amniotic fluid check": "amniotic fluid check",
    "colposcopy": "colposcopy",
    "continuous electronic fetal monitoring": "fetal monitoring",
    "saline infusion test (sit)": "saline infusion test",

    # ── Renal / urology ─────────────────────────────────
    "arthrocentesis": "arthrocentesis",
    "arthroscopy": "arthroscopy",
    "diagnostic arthroscopy": "arthroscopy",
    "diagnostic laparoscopy": "laparoscopy",
    "digital rectal exam": "digital rectal exam",
    "digital rectal exam (dre)": "digital rectal exam",
    "digital rectal examination": "digital rectal exam",
    "dried blood spot": "dried blood spot",
    "joint fluid analysis": "joint fluid analysis",
    "joint fluid test": "joint fluid analysis",
    "laparoscopy": "laparoscopy",
    "oximetry": "pulse oximetry",
    "oxygen saturation (pulse oximetry)": "pulse oximetry",
    "qsofa (quick sequential organ failure)": "sofa score",
    "skin prick test": "skin prick test",
    "sofa score": "sofa score",

    # ── Miscellaneous ───────────────────────────────────
    "arterial blood gas test": "arterial blood gas",
    "barium or air enema": "barium or air enema",
    "bone scans": "bone scans",
    "bromosulfophthalein (bsp) excretion study": "bromosulfophthalein excretion study",
    "bun/cr ratio": "bun/cr ratio",
    "cholesterol": "cholesterol",
    "computerized tomography": "computerized tomography",
    "creatine kinase-mb (ck-mb)": "ck-mb",
    "dermoscopy": "dermoscopy",
    "diagnostic mammogram": "mammography",
    "drug toxicity test": "drug toxicity test",
    "dual-energy x-ray absorptiometry": "dexa scan",
    "electrolytes & minerals": "electrolytes & minerals",
    "electroretinogram (erg)": "electroretinogram",
    "facial nerve palsy": "facial nerve palsy",
    "flow cytometry": "flow cytometry",
    "gene mutations": "gene mutations",
    "genetic evaluation": "genetic evaluation",
    "hormone receptor testing": "hormone receptor testing",
    "hormone-specific tests": "hormone-specific tests",
    "ketone test": "ketone test",
    "mammograms": "mammography",
    "microscopic: 0-5 wbc/hpf": "microscopic: 0-5 wbc/hpf",
    "nasal swab": "swab",
    "neurological & neuropsychological assessments": "neurological & neuropsychological assessments",
    "oral sodium loading test": "oral sodium loading test",
    "ph 4.5-8.0": "ph 4.5-8.0",
    "phrenic nerve stimulation test": "phrenic nerve stimulation test",
    "quantitative sensory testing": "quantitative sensory testing",
    "random plasma": "random plasma",
    "rapid immunoassays": "rapid immunoassays",
    "scintigraphy": "scintigraphy",
    "serum free light chain (sflc) quantification": "serum free light chain",
    "sinus cultures": "sinus cultures",
    "skin turgor test": "skin turgor test",
    "spect scans": "spect scans",
    "the hepatic venous pressure gradient (hvpg)": "hepatic venous pressure gradient",
    "throat swab": "swab",
    "tissue and fluid analysis": "tissue and fluid analysis",
    "tzanck smear": "tzanck smear",
    "venous plethysmography": "venous plethysmography",
    "videofluoroscopic swallow study": "videofluoroscopic swallow study",
    "wood's lamp exam": "wood's lamp examination",
    "x-rays": "x-rays",

    }
    return LAB_SYNONYMS.get(name, name)

def parse_numeric_range(range_str):
    """Enhanced to handle <, >, ≤"""
    if not isinstance(range_str, str):
        return None, None
    cleaned = range_str.replace(',', '').strip()
    # low-high
    match = re.search(r'(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)', cleaned)
    if match:
        low = float(match.group(1))
        high = float(match.group(2))
        return (low, high) if low < high else (None, None)
    # less than
    match = re.search(r'[<≤]\s*(\d+(?:\.\d+)?)', cleaned)
    if match:
        return (None, float(match.group(1)))
    # greater than
    match = re.search(r'>\s*(\d+(?:\.\d+)?)', cleaned)
    if match:
        return (float(match.group(1)), None)
    return None, None

def parse_sub_ranges(normal_range):
    """
    Parse (low, high) from sub‑ranges in a normal_range string.
    Returns dict {normalized_lab_name: (low, high)}.
    """
    if not normal_range or not isinstance(normal_range, str):
        return {}
    sub_ranges = {}
    # Same pattern but capture low and high as well
    pattern = r'([A-Za-z][A-Za-z\s]*?)\s*:?\s*([\d,]+\.?[\d]*)\s*-\s*([\d,]+\.?[\d]*)'
    for m in re.finditer(pattern, normal_range):
        key = m.group(1).strip()
        try:
            low = float(m.group(2).replace(',', ''))
            high = float(m.group(3).replace(',', ''))
            if low < high:
                normalized_key = normalize_lab_name(key)
                sub_ranges[normalized_key] = (low, high)
        except:
            continue
    return sub_ranges

def extract_sub_lab_names(normal_range):
    """
    Extract lab names from a normal_range string that contains sub‑ranges.
    Supports:
      - 'WBC: 4,500-11,000'  (colon)
      - 'Sodium 135-145'      (space, no colon)
      - 'Total Protein: 6.0-8.3' (multi‑word)
    """
    if not normal_range or not isinstance(normal_range, str):
        return []
    # Pattern: lab name (letters and spaces, non‑greedy), optional colon, then a numeric range like "100-200"
    pattern = r'([A-Za-z][A-Za-z\s]*?)\s*:?\s*[\d,]+\.?[\d]*\s*-\s*[\d,]+\.?[\d]*'
    matches = re.findall(pattern, normal_range)
    cleaned = [m.strip() for m in matches]          # remove leading/trailing spaces
    return [normalize_lab_name(m) for m in cleaned]

def scale_numeric_value(value, low_range, high_range):
    s = str(value).strip()
    match = re.match(r'^([\d,]+\.?\d*)', s.replace(' ', ''))
    if match:
        s = match.group(1).replace(',', '')
    try:
        num = float(s)
    except:
        return None
    if low_range > 1000 and num < 1000:
        num = num * 1000
    return num

def is_imaging_test(lab_name):
    imaging_keywords = ['x-ray', 'ct scan', 'mri', 'ultrasound', 'echocardiogram', 'ecg', 'ekg',
                        'electrocardiogram', 'echo', 'angiography', 'stress test', 'chest x']
    return any(kw in lab_name.lower() for kw in imaging_keywords)

def is_normal_imaging_value(value):
    normal_phrases = ['negative', 'normal', 'no abnormality', 'unremarkable', 'without abnormalities']
    if not isinstance(value, str):
        value = str(value)
    return any(phrase in value.lower() for phrase in normal_phrases)

# ------------------- Categorical labs (no numeric ranges) -------------------
CATEGORICAL_LAB_NAMES = {
    "x-ray", "ct", "tee", "ecg", "mri", "cardiac cath", "carotid doppler", "echocardiogram",
    "urine culture", "cta chest w/ contrast", "cta", "cta head", "ultrasound",
    "ct maxillofacial", "mri spine thoracic", "mri brain", "mri spine cervical",
    "xr pelvis", "ct pelvis", "ct abdomen", "ct abdomen and pelvis w/ contrast",
    "pelvic ultrasound", "pcr"
}

def is_categorical_test(lab_name):
    """Return True if the lab name is in the categorical list (case-insensitive)."""
    if not isinstance(lab_name, str):
        return False
    return lab_name.lower().strip() in CATEGORICAL_LAB_NAMES

def is_categorical_normal(value):
    """
    Determine if a categorical result is normal.
    Normal: 'Negative' or 'Ordered' or 'Normal'
    Abnormal: 'Positive' or 'Abnormal'
    Other values: treat as abnormal (or could raise warning)
    """
    if not isinstance(value, str):
        value = str(value)
    val_lower = value.lower().strip()
    if val_lower in ["negative", "ordered", "normal"]:
        return True
    elif val_lower in ["positive", "abnormal"]:
        return False
    else:
        # Unknown categorical value – treat as abnormal (or print warning)
        print(f"Warning: Unknown categorical value '{value}' – treating as abnormal")
        return False

PANEL_KEYWORDS = {
    "complete blood count": ["White Blood Cells", "Red Blood Cells", "Hemoglobin", "Hematocrit",
                             "Platelets", "RDW", "Mean Corpuscular Volume", "Mean Corpuscular Hemoglobin",
                             "Mean Corpuscular Hemoglobin Concentration", "MPV"],
    "cbc": ["White Blood Cells", "Red Blood Cells", "Hemoglobin", "Hematocrit",
            "Platelets", "RDW", "Mean Corpuscular Volume", "Mean Corpuscular Hemoglobin",
            "Mean Corpuscular Hemoglobin Concentration", "MPV"],
    "basic metabolic panel": ["Sodium", "Potassium", "Chloride", "Carbon Dioxide",
                              "Glucose", "BUN", "Creatinine", "Calcium"],
    "bmp": ["Sodium", "Potassium", "Chloride", "Carbon Dioxide", "Glucose", "BUN", "Creatinine", "Calcium"],
    "comprehensive metabolic panel": ["Sodium", "Potassium", "Chloride", "Carbon Dioxide", "Glucose",
                                      "BUN", "Creatinine", "Calcium", "Albumin", "Total Protein",
                                      "AST", "ALT", "Alkaline Phosphatase", "Total Bilirubin"],
    "cmp": ["Sodium", "Potassium", "Chloride", "Carbon Dioxide", "Glucose", "BUN", "Creatinine",
            "Calcium", "Albumin", "Total Protein", "AST", "ALT", "Alkaline Phosphatase", "Total Bilirubin"],
    "liver function tests": ["ALT", "AST", "Alkaline Phosphatase", "Total Bilirubin", "Albumin", "Total Protein"],
    "lfts": ["ALT", "AST", "Alkaline Phosphatase", "Total Bilirubin", "Albumin", "Total Protein"],
    "lipid panel": ["Total Cholesterol", "LDL Cholesterol", "HDL Cholesterol", "Triglycerides"],
    "coagulation": ["PT", "INR", "PTT"],
    "hemoglobin a1c": ["Hemoglobin A1c"], "hba1c": ["Hemoglobin A1c"],
    "thyroid stimulating hormone": ["TSH"], "tsh": ["TSH"],
    "heart rate": ["Heart Rate"],
    "blood pressure": ["Systolic BP", "Diastolic BP"],
    "respiratory rate": ["Respiratory Rate"],
    "temperature": ["Temperature"],
    "oxygen saturation": ["Oxygen Saturation"],
    "bmi": ["BMI"],
    "brain natriuretic peptide": ["BNP"],
    "arterial blood gas": ["pH", "PaCO2", "PaO2", "Bicarbonate"],
    "abg": ["pH", "PaCO2", "PaO2", "Bicarbonate"],
}

def expand_lab_names(lab_name, normal_range=None):
    if normal_range:
        sub_labs = extract_sub_lab_names(normal_range)
        if sub_labs:
            return sub_labs
    lab_lower = lab_name.lower().strip()
    # Exact match on panel keywords (whole word, not substring)
    for keyword, components in PANEL_KEYWORDS.items():
        if lab_lower == keyword:          # ✅ exact match
            return [normalize_lab_name(c) for c in components]
    return [lab_name]

def extract_lab_names_via_regex(text):
    pattern = r'"lab_name":\s*"([^"]+)"'
    matches = re.findall(pattern, text)
    if not matches:
        pattern = r"lab_name:\s*'([^']+)'"
        matches = re.findall(pattern, text)
    return matches

# ------------------- Build  icd_range_map from raw dataframe -------------------
def build_med_lab_database(df):
    db = {}
    icd_range_map = {}
    parse_errors = 0
    regex_fallbacks = 0
    skipped = 0

    for _, row in df.iterrows():
        icd_raw = row['icd_code']
        if pd.isna(icd_raw):
            skipped += 1
            continue
        icd = ensure_decimal(str(icd_raw).strip())
        disease = str(row['disease_name']).strip() if pd.notna(row['disease_name']) else ''
        if not disease:
            skipped += 1
            continue

        lab_tests_raw = row['lab_tests']
        if pd.isna(lab_tests_raw):
            db[icd] = {'disease': disease, 'labs': set()}
            continue

        lab_tests_raw = str(lab_tests_raw)
        lab_list = []
        try:
            parsed = json.loads(lab_tests_raw)
            lab_list = parsed if isinstance(parsed, list) else [parsed]
        except:
            try:
                parsed = ast.literal_eval(lab_tests_raw)
                lab_list = parsed if isinstance(parsed, list) else [parsed]
            except:
                lab_names = extract_lab_names_via_regex(lab_tests_raw)
                if lab_names:
                    lab_list = [{"lab_name": name} for name in lab_names]
                    regex_fallbacks += 1
                else:
                    parse_errors += 1

        if not lab_list:
            db[icd] = {'disease': disease, 'labs': set()}
            continue

        relevant = set()
        for lab in lab_list:
            lab_name = lab.get('lab_name', '').strip() if isinstance(lab, dict) else str(lab).strip()
            normal_range = lab.get('normal_range', None) if isinstance(lab, dict) else None
            if not lab_name:
                continue

            expanded = expand_lab_names(lab_name, normal_range)
            sub_ranges = parse_sub_ranges(normal_range) if normal_range else {}

            for exp in expanded:
                normalized = normalize_lab_name(exp)
                relevant.add(normalized)
                if normalized in sub_ranges:
                    icd_range_map[(icd, normalized)] = sub_ranges[normalized]
                elif normal_range and (icd, normalized) not in icd_range_map:
                    low, high = parse_numeric_range(normal_range)
                    if low is not None or high is not None:
                        icd_range_map[(icd, normalized)] = (low, high)

        db[icd] = {'disease': disease, 'labs': relevant}

    print(f"Built {len(db)} ICD entries. Skipped {skipped} rows. Range map size: {len(icd_range_map)}")
    return db, icd_range_map

# ------------------- Load extracted labs into patient_labs dict -------------------
def build_patient_labs(df):
    patient_labs = defaultdict(list)
    for _, row in df.iterrows():
        chart_no = row['Chart_No']
        labs_json = row['Extracted_Labs']
        if pd.notna(labs_json) and labs_json != '' and labs_json != '[]':
            try:
                data = json.loads(labs_json)
                labs_list = data.get('labs', [])
                for lab in labs_list:
                    lab['normalized_name'] = normalize_lab_name(lab.get('lab_name', ''))
                    patient_labs[chart_no].append(lab)
            except:
                print(f"Warning: Could not parse labs for chart {chart_no}")
    return patient_labs

# ------------------- Load chart text -------------------
def build_chart_texts(df):
    patients = defaultdict(str)
    for _, row in df.iterrows():
        chart_no = row['Chart_No']
        all_text = ''
        for col in df.columns:
            if col not in ['Chart_No', 'Visit', 'DOS', 'Visit_Time', 'ALLERGIES/ALLERGY']:
                val = str(row.get(col, ''))
                all_text += ' ' + val
        patients[chart_no] += ' ' + all_text
    for chart_no in patients:
        patients[chart_no] = patients[chart_no].lower()
    return patients

# ------------------- Load ICD classification map -------------------
def build_icd_category_map(df):
    cat_map = {}
    for _, row in df.iterrows():
        rng = str(row['ICD 10 Code range']).strip()
        cat = row['Category']
        if '-' in rng:
            s, e = rng.split('-')
            cat_map[(s.strip(), e.strip())] = cat
        else:
            cat_map[(rng, rng)] = cat
    return cat_map


def classify_icd(icd, icd_category_map1, icd_category_map2):
    icd = str(icd).strip().replace('.', '')
    prefix = icd[:3]

    # Search Sheet1 first
    for (s, e), cat in icd_category_map1.items():
        s = str(s).replace('.', '')
        e = str(e).replace('.', '')

        if s == e:
            if s == icd:
                return cat
        else:
            if s <= prefix <= e:
                return cat

    # If not found, search Sheet2
    for (s, e), cat in icd_category_map2.items():
        s = str(s).replace('.', '')
        e = str(e).replace('.', '')

        if s == e:
            if s == icd:
                return cat
        else:
            if s <= prefix <= e:
                return cat

    return "Unknown"

# ------------------- Parse ICD team extraction -------------------
def parse_extraction_records(df):
    records = []
    cols = df.columns.tolist()
    if 'unique_icd' in cols:
        icd_col = 'unique_icd'
        split_on = '\n'
    elif 'ICD_Codes' in cols:
        icd_col = 'ICD_Codes'
        split_on = ','
    else:
        raise ValueError(f"Cannot find ICD column. Columns: {cols}")
    for _, row in df.iterrows():
        chart_no = row['chart_no']
        raw = str(row.get(icd_col, '')).strip()
        if not raw or raw in ('nan', 'NAN', ''):
            continue
        for code in raw.split(split_on):
            code = code.strip()
            if code:
                records.append({'chart_no': chart_no, 'icd_code': ensure_decimal(code)})
    return records

# ------------------- Process the raw data -------------------
print("Processing raw data...")
icd_category_map1 = build_icd_category_map(raw_icd_class_df1)
icd_category_map2 = build_icd_category_map(raw_icd_class_df2)  
med_lab_db, icd_range_map = build_med_lab_database(raw_lab_database_df)
patient_labs = build_patient_labs(raw_extracted_labs_df)
chart_texts = build_chart_texts(raw_chart_data_df)
extraction_records = parse_extraction_records(raw_extraction_df)

print("All data processed and ready.")

Processing raw data...
Built 64780 ICD entries. Skipped 0 rows. Range map size: 222750
All data processed and ready.


In [ ]:
# Cell 3: Define the main function for extracting relevant lab summaries

# ------------------- Updated get_relevant_lab_summary (strict, no fallback) -------------------
from datetime import datetime

def get_relevant_lab_summary(chart_no, patient_labs_dict, relevant_lab_names, icd_range_map, icd):
    def is_within_range(value, low, high):
        if low is not None and value < low:
            return False
        if high is not None and value > high:
            return False
        return True

    # Collect all labs for this patient
    all_labs = patient_labs_dict.get(chart_no, [])
    
    # Group by normalized lab name, keep all measurements (with datetime)
    lab_measurements = {}
    for lab in all_labs:
        norm_name = lab.get('normalized_name', '')
        if not norm_name:
            continue
        if norm_name not in lab_measurements:
            lab_measurements[norm_name] = []
        lab_measurements[norm_name].append({
            'lab_name_raw': lab.get('lab_name', 'Unknown'),
            'value': lab.get('lab_value', ''),
            'datetime': lab.get('datetime', ''),
            'flag': lab.get('lab_label', '')
        })

    normal = []
    abnormal = []

    for norm_name in relevant_lab_names:
        measurements = lab_measurements.get(norm_name, [])
        if not measurements:
            continue  # no measurements for this lab
        
        # Get ICD-specific numeric range (low or high may be None)
        low, high = icd_range_map.get((icd, norm_name), (None, None))

        # --- If no numeric range, check if it's a categorical lab ---
        if low is None and high is None:
            if norm_name in CATEGORICAL_LAB_NAMES:
                # Process categorically
                abnormal_measurements = []
                normal_measurements = []
                for m in measurements:
                    val = str(m['value']).lower().strip()
                    if val in ['negative', 'ordered', 'normal']:
                        normal_measurements.append(m)
                    elif val in ['positive', 'abnormal']:
                        abnormal_measurements.append(m)
                    else:
                        # Unknown categorical value – treat as abnormal (or you could raise a warning)
                        print(f"Warning: Unknown categorical value '{m['value']}' for lab '{norm_name}' in chart {chart_no} – treating as abnormal")
                        abnormal_measurements.append(m)

                # Decision (same as numeric)
                if abnormal_measurements:
                    chosen = abnormal_measurements[0]
                    abnormal.append(f"{chosen['lab_name_raw']}:{chosen['value']}")
                elif normal_measurements:
                    def parse_dt(dt_str):
                        try:
                            return datetime.strptime(dt_str, "%Y-%m-%d %H:%M")
                        except:
                            return datetime.min
                    latest = max(normal_measurements, key=lambda x: parse_dt(x['datetime']))
                    normal.append(f"{latest['lab_name_raw']}:{latest['value']}")
                continue   # done for this categorical lab
            else:
                # No numeric range and not categorical – skip
                print(f"WARNING: No numeric range for ICD {icd}, lab '{norm_name}' in chart {chart_no}")
                continue

        # --- Numeric processing (original) ---
        abnormal_measurements = []
        normal_measurements = []
        for m in measurements:
            raw_value = m['value']
            num_value = scale_numeric_value(raw_value, low if low else 0, high if high else 0)
            if num_value is None:
                # If value cannot be converted, use flag as fallback
                if m['flag'] in ['L', 'LL', 'H', 'HH']:
                    abnormal_measurements.append(m)
                else:
                    normal_measurements.append(m)
                continue

            if is_within_range(num_value, low, high):
                normal_measurements.append(m)
            else:
                abnormal_measurements.append(m)

        if abnormal_measurements:
            chosen = abnormal_measurements[0]
            abnormal.append(f"{chosen['lab_name_raw']}:{chosen['value']}")
        elif normal_measurements:
            def parse_dt(dt_str):
                try:
                    return datetime.strptime(dt_str, "%Y-%m-%d %H:%M")
                except:
                    return datetime.min
            latest = max(normal_measurements, key=lambda x: parse_dt(x['datetime']))
            normal.append(f"{latest['lab_name_raw']}:{latest['value']}")

    normal.sort()
    abnormal.sort()
    normal_str = f"Normal({', '.join(normal)})" if normal else "Normal()"
    abnormal_str = f"Abnormal({', '.join(abnormal)})" if abnormal else "Abnormal()"
    return normal_str, abnormal_str
        

# ------------------- Ambiguous wording helpers for Query -------------------
AMBIGUOUS_KEYWORDS = ['previous condition', 'previously treated', 'treated', 'resolved']

def contains_ambiguous_keywords(text):
    return any(kw in text for kw in AMBIGUOUS_KEYWORDS)

def get_first_ambiguous_snippet(text, window=150):
    text_lower = text.lower()
    for kw in AMBIGUOUS_KEYWORDS:
        pos = text_lower.find(kw)
        if pos != -1:
            start = max(0, pos - window)
            end = min(len(text), pos + len(kw) + window)
            snippet = text[start:end].replace('\n', ' ').strip()
            if len(snippet) > 300:
                snippet = snippet[:300] + '...'
            return f"...{snippet}..."
    return ""

def is_condition_near_ambiguous(text, condition_name, icd_code, window=200):
    text_lower = text.lower()
    cond_lower = condition_name.lower()
    icd_lower = icd_code.lower()
    stopwords = {'unspecified', 'other', 'unspec', 'without', 'with', 'chronic', 'acute', 'not', 'elsewhere', 'classified', 'type', 'stage', 'and'}
    tokens = re.split(r'[\s,()\-/]+', cond_lower)
    significant_words = {t for t in tokens if len(t) > 3 and t not in stopwords}
    significant_words.add(cond_lower)
    for kw in AMBIGUOUS_KEYWORDS:
        kw_pos = text_lower.find(kw)
        if kw_pos == -1:
            continue
        start = max(0, kw_pos - window)
        end = min(len(text_lower), kw_pos + window)
        segment = text_lower[start:end]
        if icd_lower in segment:
            return True
        for word in significant_words:
            if word in segment:
                return True
    return False

def should_keep_combination_code(icd, abnormal_codes):
    """
    Determine if a combination ICD code should be kept in the Keep column
    based on abnormal lab status of dependent codes.
    Returns True if it should remain in Keep, False otherwise.
    """
    # Define dependency rules
    # Each rule: 'code': (required_codes, optional_range_check)
    # required_codes: list of ICD codes that must be in abnormal_codes
    # range_check: function that takes abnormal_codes and returns True if any code in range is abnormal
    rules = {
        "I11.0": {
            "required": ["I10"],
            "range_check": lambda ac: any(c for c in ac if c >= "I50.1" and c <= "I50.9")
        },
        "I12.0": {
            "required": ["I10"],
            "range_check": lambda ac: any(c for c in ac if c in ["N18.5", "N18.6"])
        },
        "I12.9": {
            "required": ["I10"],
            "range_check": lambda ac: any(c for c in ac if c in ["N18.9", "N18.1", "N18.2", "N18.30", "N18.31", "N18.32", "N18.4"])
        },
        "I13.0": {
            "required": ["I10", "I50.9", "N18.9"],
            "range_check": None
        },
        "I13.10": {
            "required": ["I11.9"],
            "range_check": lambda ac: any(c for c in ac if c in ["N18.9", "N18.1", "N18.2", "N18.30", "N18.31", "N18.32", "N18.4"])
        },
        "I13.11": {
            "required": ["I11.9"],
            "range_check": lambda ac: any(c for c in ac if c in ["N18.5", "N18.6"])
        },
        "I13.2": {
            "required": ["I10"],
            "range_check": lambda ac: any(c for c in ac if c >= "I50.1" and c <= "I50.9") and any(c for c in ac if c in ["N18.5", "N18.6"])
        },
    }

    if icd not in rules:
        return True   # not a combination code, keep normally

    rule = rules[icd]
    # Check required codes are in abnormal_codes
    for req in rule["required"]:
        if req not in abnormal_codes:
            return False
    # If range_check exists, check it
    if rule["range_check"] is not None:
        if not rule["range_check"](abnormal_codes):
            return False
    return True

# ------------------- Main processing (with special handling for R,V,W,X,Y,Z codes) -------------------
groups = defaultdict(list)
for rec in extraction_records:
    groups[rec['chart_no']].append(rec['icd_code'])

results = []
for chart_no, icd_codes in groups.items():
    chart_text = chart_texts.get(chart_no, '')
    chart_has_ambiguous = contains_ambiguous_keywords(chart_text)

    keep_tags = []
    discard_tags = []
    all_icds = []
    all_diag_tags = []
    abnormal_tags = []
    query_conditions = []

    for icd in icd_codes:
        # Check if ICD starts with R, V, W, X, Y, Z (case-insensitive, but ICD codes are uppercase)
        if icd and icd[0] in ['R', 'V', 'W', 'X', 'Y', 'Z']:
            # Special handling: no lab processing, directly add to Keep, skip diagnosis tag and abnormal summary
            keep_tags.append(icd)
            all_icds.append(icd)        # still appears in ICD_team_codes column
            # Do NOT add to all_diag_tags, abnormal_tags, query_conditions
            continue

        # Normal processing for other ICDs
        entry = med_lab_db.get(icd) or {'disease': 'code_not_found_in_database', 'labs': set()}
        disease = entry['disease']
        relevant = entry['labs']

        normal_str, abnormal_str = get_relevant_lab_summary(chart_no, patient_labs, relevant, icd_range_map, icd)

        # category = classify_icd(icd, icd_category_map)
        category = classify_icd(icd, icd_category_map1, icd_category_map2)
        diag_tag = f"{icd}:{disease} {{ {normal_str} | {abnormal_str} | {category} }}"
        all_icds.append(icd)
        all_diag_tags.append(diag_tag)

        has_abnormal = abnormal_str != "Abnormal()"
        if has_abnormal:
            abnormal_tags.append(f"{icd}:{disease} {{ {abnormal_str} }}")
            keep_tags.append(icd)
        else:
            discard_tags.append(icd)

        if chart_has_ambiguous and has_abnormal and disease != 'code_not_found_in_database' and is_condition_near_ambiguous(chart_text, disease, icd):
            query_conditions.append(f"{icd}:{disease}")
        
    # Convert keep_tags to a set for efficient lookups
    abnormal_codes = set(keep_tags)

    # Apply combination rules: remove combination codes that don't meet dependencies
    filtered_keep = []
    for icd in keep_tags:
        if should_keep_combination_code(icd, abnormal_codes):
            filtered_keep.append(icd)
    keep_tags = filtered_keep

    # Build output row
    icd_team_codes = "\n".join(all_icds)
    diagnosis_tag = "\n".join(all_diag_tags)
    abnormal_summary = "\n".join(abnormal_tags)
    keep_str = "\n".join(keep_tags)
    discard_str = "\n".join(discard_tags)

    if chart_has_ambiguous and query_conditions:
        snippet = get_first_ambiguous_snippet(chart_text)
        query = f"Patient has abnormal labs and documentation contains ambiguous wording ('previous', 'treated', 'resolved') for the following conditions:\n" + \
                "\n".join(query_conditions) + f"\nAmbiguous context: {snippet}\nPlease review and rule-in/out each condition."
    else:
        query = ""

    results.append({
        'Chart_number': chart_no,
        'ICD_team_codes': icd_team_codes,
        'Diagnosis_tag': diagnosis_tag,
        'Abnormal_Labs_Summary': abnormal_summary,
        'Keep': keep_str,
        'Discard': discard_str,
        'Query': query,
    })

df_out = pd.DataFrame(results)
OUTPUT_FILE = 'ICD_Lab_Correlation_&_Query_Output_18_6_26.xlsx'
df_out.to_excel(OUTPUT_FILE, index=False)
print(f"\n✅ Output saved to {OUTPUT_FILE}")
print(f"Total charts processed: {len(df_out)}")


✅ Output saved to ICD_Lab_Correlation_&_Query_Output_18_6_26.xlsx
Total charts processed: 20
